# Low-Level Hardware Bit-Banging and Glitching with `scope.bitbanger`

This notebook teaches the basics of using `scope.bitbanger`.

In other notebooks we teach how `scope.bitbanger` can be made to speak [SWD](10%20-%20Husky%20Hardware%20SWD%20Bitbanging.ipynb) and [1-wire](11%20-%20Husky%20Hardware%201-Wire%20Bitbanging.ipynb).

If your use case fits into what's covered there, great; if not, or if you want to use `scope.bitbanger` on a totally different protocol, then you'll need to really understand how the bit-banger works, so that you can build your bit-banging from the very bottom up; this notebook teaches you how.

Going through this notebook will teach you all the possibilities and limitations of `scope.bitbanger`.

In [ ]:
SCOPE="OPENADC"
PLATFORM="CWHUSKY"

In [ ]:
%run "../../Setup_Scripts/Setup_Generic.ipynb"

Let's begin by reproducing the example shown in the `scope.bitbanger` class documentation:

In [ ]:
scope.bitbanger?

In [ ]:
scope.bitbanger.data_pin = 'USERIO_D0'
scope.bitbanger.clock_pin = 'USERIO_D1'
scope.bitbanger.continuous_clk = False
scope.bitbanger.inactive_data = 0
scope.bitbanger.inactive_state = 'high_z'
scope.bitbanger.drive_edge = 'rising'
scope.bitbanger.check_edge = 'falling'
scope.bitbanger.trigger_en = True
scope.bitbanger.clk_div = 4
scope.bitbanger.num_bits = 5

scope.bitbanger.pattern_data = [0, 1, 0, 1, 0]
scope.bitbanger.trig_bits    = [0, 0, 0, 1, 0]
scope.bitbanger.pattern_hiz  = [0, 0, 0, 0, 0]
scope.bitbanger.pattern_en   = [0, 0, 0, 0, 0]
scope.bitbanger.record_en    = [0, 0, 0, 0, 0]

The documentation tells us to expect this:

```
               ┏─┐ ┏─┐ ┏─┐ ┏─┐ ┏─┐ ┏─┐ ┏─┐ ┏─┐ ┏─┐ ┏─┐ ┏─┐ ┏─┐ ┏─┐ ┏─┐ ┏─┐ ┏─┐ ┏─┐ ┏─┐ ┏─┐ ┏─┐ ┏─┐ ┏─┐ 
    ADC clock: ┛ └─┛ └─┛ └─┛ └─┛ └─┛ └─┛ └─┛ └─┛ └─┛ └─┛ └─┛ └─┛ └─┛ └─┛ └─┛ └─┛ └─┛ └─┛ └─┛ └─┛ └─┛ └─
               ____╱              ╲╱              ╲╱              ╲╱              ╲╱              ╲____
    bit count:     ╲    bit 0     ╱╲    bit 1     ╱╲    bit 2     ╱╲    bit 3     ╱╲    bit 4     ╱    
                   ┌───────┐       ┌───────┐       ┌───────┐       ┌───────┐       ┌───────┐           
    clock out: ────┘       └───────┘       └───────┘       └───────┘       └───────┘       └───────────
               ____┐               ┌───────────────┐               ┌───────────────┐               ____
    data out :     └───────────────┘               └───────────────┘               └───────────────    
                                                                           ┌───────────────┐           
    trigger  : ────────────────────────────────────────────────────────────┘               └───────────
```

Let's set up `scope.LA` to capture what actually gets driven out:

In [ ]:
scope.LA.enabled = True
scope.LA.capture_group = 'USERIO 20-pin'
scope.LA.trigger_source = 'rising_userio_d1'
scope.LA.capture_depth = 100
scope.LA.oversampling_factor = 4

Finally, remember that `scope.bitbanger` is clocked by the **ADC** clock; let's make that the same as the `scope.LA` source clock, for simplicity:

In [ ]:
scope.clock.adc_mul = 1

In [ ]:
scope.LA.arm()
scope.bitbanger.go()

In [ ]:
raw = scope.LA.read_capture_data()
bb_data = scope.LA.extract(raw, 0)
bb_clock = scope.LA.extract(raw, 1)

In [ ]:
from bokeh.plotting import figure, show
from bokeh.resources import INLINE
from bokeh.io import output_notebook
output_notebook(INLINE)
p = figure(width=600, height=200)
xrange = list(range(len(bb_data)))
p.line(xrange, bb_data + 2, line_color='blue')
p.line(xrange, bb_clock + 0, line_color='green')
show(p)

Now let's change the drive edge to 'falling'. Note that we can simply call `scope.bitbanger.go()` without having to re-specify all the properties that we aren't changing:

In [ ]:
scope.bitbanger.drive_edge = 'falling'
scope.LA.arm()
scope.bitbanger.go()
raw = scope.LA.read_capture_data()
bb_data_falling = scope.LA.extract(raw, 0)

In [ ]:
p = figure(width=600, height=200)
p.line(xrange, bb_data_falling + 4, line_color='red')
p.line(xrange, bb_data + 2, line_color='blue')
p.line(xrange, bb_clock + 0, line_color='green')
show(p)

# Triggering
What about `scope.bitbanger`'s triggering capability?

Through `scope.bitbanger.check_edge`, you can control whether a trigger is asserted on the rising or falling edge of the `scope.bitbanger` clock line.

But how can we see it when `scope.LA` doesn't let us capture it? Actually, it does... when you use `scope.userio` pins for bit-banging, its other pins remain available to you.

By setting `scope.userio` as follows, we can get the trigger output on USERIO D3:

In [ ]:
scope.userio.mode = 'fpga_debug'
scope.userio.fpga_mode = 17
print(scope.userio)

One more thing: in order to get a trigger out of `scope.bitbanger`, we must choose it as the trigger module:

In [ ]:
scope.trigger.module = 'bitbanger'

In [ ]:
scope.LA.arm()
scope.bitbanger.go()
raw = scope.LA.read_capture_data()
bb_data_falling = scope.LA.extract(raw, 0)
trigger = scope.LA.extract(raw, 3)

In [ ]:
p = figure(width=600, height=200)
p.line(xrange, bb_data_falling + 4, line_color='red')
p.line(xrange, bb_data + 2, line_color='blue')
p.line(xrange, bb_clock + 0, line_color='green')
p.line(xrange, trigger - 2, line_color='black')
show(p)

Experiment with all four combinations of `scope.bitbanger.drive_edge` and `scope.bitbanger.check_edge` to see what they look like.

# Recording and/or Checking Data
`scope.bitbanger.check_edge` controls not only the clock edge of the trigger output, but also the clock edge that is used to sample the data line.

This can be really useful when the data line is bidirectional. 

Husky provides two means of checking what is driven on the data line:
1. With `scope.bitbanger.pattern_en`, you can check whether the line is driven with the values in `scope.bitbanger.pattern_data` at specific times. This is useful when a known response is expected from the target; you can then look up `scope.bitbanger.matched` to know whether the expected data was seen. You can check the full length of the bit-banged `pattern_data` or a subset.
2. With `scope.bitbanger.record_en`, you can record what is driven on the line at specific times. This is useful when you don't necessarily know what the target will respond with, and you wish to record it. However you are limited in the number of bits that can be recorded; the maximum is `scope.bitbanger.max_record`.

For examples of recording data from actual targets, look at our `scope.bitbanger.swd` and `scope.bitbanger.onewire`: SWD and 1-wire are both examples of a bidirectional I/O line.

These are demo'd in our separate [SWD](10%20-%20Husky%20Hardware%20SWD%20Bitbanging.ipynb) and [1-wire](11%20-%20Husky%20Hardware%201-Wire%20Bitbanging.ipynb) notebooks.

Here we don't have a target attached, but we can still illustrate the timing when Husky is the only driver. Again, we'll use `scope.userio`'s FPGA debug mode to see how this works.

In [ ]:
scope.bitbanger.drive_edge = 'rising'
scope.bitbanger.check_edge = 'rising'

scope.bitbanger.pattern_data = [0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0]
scope.bitbanger.trig_bits    = [0]*16
scope.bitbanger.pattern_hiz  = [0]*16
scope.bitbanger.pattern_en   = [1, 0]*8
scope.bitbanger.record_en    = [0, 1]*8

scope.bitbanger.num_bits = len(scope.bitbanger.pattern_data)

In [ ]:
scope.LA.capture_depth = 250

In [ ]:
scope.LA.arm()
scope.bitbanger.go()
raw = scope.LA.read_capture_data()
bb_data = scope.LA.extract(raw, 0)
bb_clock = scope.LA.extract(raw, 1)
bitrecord = scope.LA.extract(raw, 2)
pattern_en = scope.LA.extract(raw, 4)

In [ ]:
p = figure(width=900, height=200)

xrange = list(range(len(bb_data)))

p.line(xrange, bb_data + 2, line_color='blue')
p.line(xrange, bb_clock + 0, line_color='green')
p.line(xrange, pattern_en - 2, line_color='black')
p.line(xrange, bitrecord - 4, line_color='red')
show(p)

The black pulses show when the `scope.bitbanger.data_pin` line is checked against `scope.bitbanger.pattern_data`.

The check is done when `patten_en` is high on the clock edge specified by `scope.bitbanger.check_edge`.

If all specified bits match, then `scope.bitbanger.matched` returns `True`:

In [ ]:
assert scope.bitbanger.matched

The red pulses show when the `scope.bitbanger.data_pin` is recorded. The value is sampled on the rising edge of `bitrecord`.

Read `scope.bitbanger.recorded_data()` to obtain the recorded data. The most significant bit is the last recorded bit:

In [ ]:
bin(scope.bitbanger.recorded_data(1))

# Glitching

With `scope.trigger.module = 'bitbanger'`, `scope.bitbanger` can issue the trigger for voltage or clock glitching -- just as with any other trigger module.

But you can also do something else that's pretty neat: instead of directing the glitch onto the HS2 clock, or the `scope.io.glitch_hp` or `scope.io.glitch_lp` MOSFETs, the glitch can be directed onto `scope.bitbanger.data_line`.

Simply configure `scope.glitch` as you normally would, then set `scope.bitbanger.glitch_mode` to one of:
* 'drive_low'
* 'drive_high'
* 'invert'

and the `scope.bitbanger.data_pin` will be affected accordingly by the glitch.

Here's an example: we'll configure `scope.bitbanger` to trigger on the last bit of its pattern, and `scope.glitch` to issue a series of short glitches after a short delay.

We'll have to crank up `scope.LA`'s oversampling so that we can see the glitches:

In [ ]:
scope.LA.capture_depth = 2500
scope.LA.oversampling_factor = 32

In [ ]:
scope.bitbanger.inactive_state = 'driven'
scope.bitbanger.glitch_mode = 'invert'
scope.bitbanger.trigger_en = True
scope.bitbanger.trig_bits[-1] = 1

scope.trigger.module = 'bitbanger'

scope.glitch.enabled = True
scope.glitch.clk_src = 'pll'
scope.glitch.trigger_src = 'ext_continuous'
scope.glitch.output = 'glitch_only'
scope.glitch.repeat = 4
scope.glitch.width = scope.glitch.phase_shift_steps//4
scope.glitch.ext_offset = 1

In [ ]:
scope.LA.arm()
scope.bitbanger.go()
raw = scope.LA.read_capture_data()
bb_data = scope.LA.extract(raw, 0)
bb_clock = scope.LA.extract(raw, 1)

In [ ]:
p = figure(width=900, height=200)

xrange = list(range(len(bb_data)))

p.line(xrange, bb_data + 2, line_color='blue')
p.line(xrange, bb_clock + 0, line_color='green')
show(p)

An interesting use for this with 1-wire targets, where the I/O data line is also the target's power supply, is to execute a very precisely timed "starvation" of the target's power supply after sending it a command.

While you could do this simply via `scope.bitbanger.pattern_data`, you might run out of bits; by using `scope.glitch` instead, you have much more freedom, and you can also time you glitch much more finely. 